# SELENE: Sea Level Near-real time Quality Control Processing

This notebook demonstrates how to use SELENE for quality control of tide gauge data.
Based on the Design & User's Guide v1.0 by Puertos del Estado.


## 1. Environment Setup
install selene with conda.  
We will be using the python from the conda environment in this notebook to work with selene. (/opt/conda/envs/selene_training/bin/python)  
Outside this environment you would install selene with conda and run the scripts within an activated environment.  

### 1.1 install Selene environement

In [ ]:
!chmod +x ./setup.sh
!./setup.sh

### 1.2 Install and load python packages

In [ ]:
import sys 
!{sys.executable} -m pip install utide python-dotenv 
print("\n✓ packages installed")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import configparser
import sys
import logging
import warnings
import subprocess
import requests
import matplotlib.pyplot as plt
import utide
import os
import configuration.constants as c
import ipywidgets as widgets
import csv 
import copy
from datetime import timedelta
from datetime import datetime
from dotenv import load_dotenv

print("\n✓ Imports succeeded")

In [ ]:
load_dotenv('./env')

## 2. Configuration Setup

### 2.1 Set up directory structure

In [ ]:
# Define base paths (modify these according to your installation)
BASE_PATH = "./"  # Update this!
DATAFILES_PATH = os.path.join(BASE_PATH, "datafiles")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs")
CONFIG_PATH = os.path.join(BASE_PATH, "configuration")

# Create directories if they don't exist
os.makedirs(DATAFILES_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CONFIG_PATH, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Datafiles path: {DATAFILES_PATH}")
print(f"Output path: {OUTPUT_PATH}")

## 3. Example Run SELENE

### 3.1 Execute SELENE Processing

In [ ]:
# Target station ID
with open(c.stationsfile) as f:
    stations = json.load(f)
TARGET_STATION = widgets.Dropdown(
    options=stations.keys(),
    description='Target Station:',
    disabled=False,
)

TARGET_STATION

In [ ]:
# Run SELENE processing
print(f"Running SELENE for station {TARGET_STATION.value}...")
print("-" * 50)

!/opt/conda/envs/selene_training/bin/python selene.py {TARGET_STATION.value} 

### 3.2 Buddy Check Processing

In [ ]:
# Buddy check requires target year as last year of data
TARGET_YEAR = widgets.IntSlider(
    value=2026,
    min=1970,
    max=2026,
    step=1,
    description='TARGET YEAR:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

TARGET_YEAR

In [ ]:
print(f"Running buddy check for station {TARGET_STATION.value}, year {TARGET_YEAR.value}...")

!/opt/conda/envs/selene_training/bin/python buddy_check.py {TARGET_STATION.value} {TARGET_YEAR.value}

### 3.3 Visualization

#### 3.3.1 Generate the graphs

In [ ]:
# Run SELENEVIS for visualization
print(f"Running SELENEVIS for station {TARGET_STATION.value}...")

!/opt/conda/envs/selene_training/bin/python selenevis.py -station {TARGET_STATION.value}

#### 3.3.2 Open the generated image

In [ ]:
from IPython.display import Image

with open(c.stationsfile) as f:
    stations = json.load(f)
    station_name = stations[TARGET_STATION.value]['name']
    
Image(filename=OUTPUT_PATH + "/SELENEQC_graphics_Station_" + station_name + ".png")

#### 3.3.3 View Output files for targeted station

In [ ]:
# List output files
print( OUTPUT_PATH)
output_files = []
if os.path.exists('./' + OUTPUT_PATH):
    output_files = os.listdir('./' + OUTPUT_PATH)
    
for file in sorted(output_files):
    if TARGET_STATION.value in file:
        filepath = os.path.join(OUTPUT_PATH, file)
        size = os.path.getsize(filepath)
        print(f"  - {file} ({size:,} bytes)")

## 4. Custom Data Integration

To add your own station data:

1. **Prepare data file**: Create ASCII .data file in `datafiles/`
2. **Update stations.json**: Add station configuration
3. **Update code_guide.csv** (if using): Add station metadata
4. **Run SELENE**: Execute with your station ID

## 4.1 Retrieve data from SLSMF

## SLSMF API 
### Authentication: 
Go to our website and request an account [here](https://ioc-sealevelmonitoring.org/api.php).  
After your account has been approved you can request an api key [here](https://ioc-sealevelmonitoring.org/api.php).  
Place your api key in the env file, in accordance to the env.example file, or place copy it below.
### Documentation
Go to [api documentation](https://api.ioc-sealevelmonitoring.org/v2/doc), for a complete overview of our endpoints.  
If you wish to try out some enpoints use the api key you just created.  


In [ ]:
print(os.getenv('API_KEY'))
APIKEY = widgets.Text(
    value=os.getenv('API_KEY'),
    placeholder='Type something',
    description='APIKEY:',
    disabled=False   
)
APIKEY

In [ ]:
print(os.getenv('API_KEY'))
APIKEY = widgets.Text(
    value=os.getenv('API_KEY'),
    placeholder='Type something',
    description='APIKEY:',
    disabled=False   
)

APIKEY

In [ ]:
station = widgets.Text(
    value='abas',
    placeholder='IOC code',
    description='IOC:',
    disabled=False   
)
sensor = widgets.Text(
    value='rad',
    description='sensor type:',
    disabled=False   
)
timestart=widgets.DatePicker(
    description='timestart',
    disabled=False
)
timestop=widgets.DatePicker(
    description='timestop',
    disabled=False
)

from ipywidgets import Box

items = [station, sensor, timestart, timestop]
box = Box(children=items)

box

In [ ]:
# Load environment variables from the .env file
loop_start = copy.deepcopy(timestart.value)
output_file = DATAFILES_PATH + '/' + station.value + '.data' 

if os.path.exists(output_file):
    os.remove(output_file)

while (timestop.value - loop_start).total_seconds() > 0:
    timestop_parameter = loop_start + timedelta(days=30)
    url = "https://api.ioc-sealevelmonitoring.org/v2/stations/"+ station.value + "/data"
    querystring = {"timestart": loop_start.strftime("%Y-%m-%d"), "timestop": timestop_parameter  }
    headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "application/json"}
    response = requests.get(url, headers=headers, params=querystring)
    print(response.url)
   
    with open(DATAFILES_PATH + '/' + station.value + '.data', "a") as f:
        for entry in json.loads(response.text):
            csvwriter = csv.writer(f, delimiter=' ', quotechar='"', quoting=csv.QUOTE_MINIMAL)
            csvwriter.writerow([station.value, entry['stime'], entry['slevel']])
    loop_start = timestop_parameter

In [ ]:
# Example: Adding a custom station
def add_custom_station(station_id, name, lat, lon, data_file):
    """
    Add a custom station to SELENE configuration
    
    Parameters:
    station_id (str): 7-digit station code
    name (str): Station name
    lat (float): Latitude
    lon (float): Longitude
    data_file (str): Path to data file
    """
    
    # Load existing configuration
    with open(stations_path, 'r') as f:
        stations = json.load(f)
    
    # Add new station
    stations[station_id] = {
        "name": name,
        "shortname": name[:4].upper(),
        "latitude": lat,
        "longitude": lon,
        "seriesfile": f"datafiles/{data_file}",
        "seriesseparator": " ",
        "seriesdatecolumns": "1,2,3",
        "seriesdateformat": "%Y%m%d%H%M%S",
        "seriesvaluecolumn": 4,
        "seriesqccolumn": 5,
        "qc_level_nsigma": 4,
        "qc_level_winsize": 200,
        "qc_level_splinedegree": 2,
        "qc_surge_nsigma": 5,
        "qc_surge_winsize": 1000,
        "qc_surge_splinedegree": 3,
        "qc_stucklimit": 5,
        "foremanharmfile": ""
    }
    
    # Save updated configuration
    with open(stations_path, 'w') as f:
        json.dump(stations, f, indent=4)
    
    print(f"✓ Added station {station_id} ({name}) to configuration")
    print(f"  Location: {lat}, {lon}")
    print(f"  Data file: {data_file}")

# Example usage (uncomment and modify):
# add_custom_station("2000001", "NEW_STATION", 45.5, -5.5, "2000001.data")

## 8. Troubleshooting

### Common Issues

**1. Module not found errors**
- Ensure you're in the SELENE directory
- Check that Python environment is activated: `conda activate selene_training`

**2. Permission denied**
- Give execution permissions: `chmod 755 foreman/predicc.e`

**3. Data format errors**
- Verify .data file format matches specifications
- Check column separators and date/time formats

**4. Missing harmonic constants**
- Set `foremanharmfile` to empty string ("") if not available
- Tide/surge analysis will be skipped

### Runtime Warnings

You may see warnings like:
```
RankWarning: Polyfit may be poorly conditioned
```
These are expected and don't prevent processing from completing.

## 9. Output Files Reference

SELENE generates several output files in the `output/` directory:

| Algorithm | Output Files | Description |
|-----------|--------------|-------------|
| Selene.py | `{code}_original_sampling_flags.out` | Original data with quality flags |
| Buddy_check.py | `{code}_original_sampling_buddy.out` | Buddy check results |
| | `{code}_hourly_slev_buddy.out` | Hourly sea levels |
| | `{code}_hourly_surge_buddy.out` | Hourly surges |
| | `{code}_hourly_tide_buddy.out` | Hourly tides |
| Selenevis.py | `{station}_*.png` | Visualization plots |

## 10. Next Steps

### Additional Resources

- **Official Documentation**: https://puertos-del-estado-medio-fisico.github.io/SELENE/
- **Copernicus Marine Service**: https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_SSH_DISCRETE_MY_013_053/description
- **Support Contact**: Dr. Begoña Pérez Gómez - bego@puertos.es

### Advanced Usage

- **Automated Processing**: Set up cron jobs with `selenelauncher.py`
- **Database Integration**: Use `selenedatadownload.py` and `selenedbconsolide.py`
- **Custom Filters**: Modify `filterhandler.py` for different filtering methods
- **Parallel Processing**: Configure multiprocessing in launcher script

---

**Note**: This notebook is based on SELENE Version 1.0 by Puertos del Estado.
For production use, ensure all configuration parameters are properly set for your
specific stations and data characteristics.

# Getting data from SLSMF
## SLSMF API 
### Authentication: 
Go to our website and request an account [here](https://ioc-sealevelmonitoring.org/api.php).  
After your account has been approved you can request an api key [here](https://ioc-sealevelmonitoring.org/api.php).  
### Documentation
Go to [api documentation](https://api.ioc-sealevelmonitoring.org/v2/doc), for a complete overview of our endpoints.  
If you wish to try out some enpoints use the api key you just created.  

In [ ]:



# Load environment variables from the .env file
load_dotenv('./env')

url = "https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/rad/tidal-harmonics"
querystring = {"datestop": "2026-05-06", "lateral_correction":"true"}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "text/csv"}
response = requests.get(url, headers=headers, params=querystring)

print(response.text)

In [ ]:

# Load environment variables from the .env file
load_dotenv('./env')

url = "https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data"
querystring = {}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "text/csv"}
response = requests.get(url, headers=headers, params=querystring)

print(response.text)

# Utide intecration 

In [ ]:
print("\n" + "="*60)
print("Notebook completed!")
print("="*60)
print("\nTo run SELENE with your data:")
print(f"  1. Place your .data file in: {DATAFILES_PATH}")
print(f"  2. Update stations.json with your station configuration")
print(f"  3. Run: /opt/conda/envs/selene_training/bin/python selene.py YOUR_STATION_ID")
print(f"  4. Visualize: /opt/conda/envs/selene_training/bin/python selenevis.py -station YOUR_STATION_ID")

In [ ]:
import sys 
!{sys.executable} -m pip install utide

In [ ]:
%matplotlib inline


print(utide.__version__)

Look at the data file to see what structure it has.

In [ ]:
datafile = os.path.join(DATAFILES_PATH, "can1998.dtf")
with open(datafile) as f:
    lines = f.readlines()

print("".join(lines[:5]))

It looks like the fields are seconds, year, month, day, hour, elevation, flag.  We need a date parser function to combine the date and time fields into a single value to be used as the datetime index.

In [ ]:
names = ["seconds", "year", "month", "day", "hour", "elev", "flag"]

obs = pd.read_csv(
    datafile,
    names=names,
    skipinitialspace=True,
    delim_whitespace=True,
    na_values="9.990",
)


date_cols = ["year", "month", "day", "hour"]
index = pd.to_datetime(obs[date_cols])
obs = obs.drop(date_cols, axis=1)
obs.index = index

obs.head(5)

Although there are no elevations marked bad via special value, which should be `nan` after reading the file, the flag value of 2 indicates the values are unreliable, so we will mark them with `nan`, calculate the deviations of the elevations from their mean (stored in a new column called "anomaly"), and then interpolate to fill in the `nan` values in the anomaly.

In [ ]:
bad = obs["flag"] == 2
corrected = obs["flag"] == 1

obs.loc[bad, "elev"] = np.nan
obs["anomaly"] = obs["elev"] - obs["elev"].mean()
obs["anomaly"] = obs["anomaly"].interpolate()
print(f"{bad.sum()} points were flagged 'bad' and interpolated")
print(f"{corrected.sum()} points were flagged 'corrected' and left unchanged")

Now we can call solve to obtain the coefficients.

In [ ]:
coef = utide.solve(
    obs.index,
    obs["anomaly"],
    lat=-25,
    method="ols",
    conf_int="MC",
    verbose=False,
)

The amplitudes and phases from the fit are now in the `coef` data structure (a Bunch), which can be used directly in the `reconstruct` function to generate a hindcast or forecast of the tides at the times specified in the `time` array.

In [ ]:
print(coef)

In [ ]:
tide = utide.reconstruct(obs.index, coef, verbose=False)

The output from the reconstruction is also a Bunch:

In [ ]:
print(tide.keys())

In [ ]:
t = obs.index.to_pydatetime()

fig, (ax0, ax1, ax2) = plt.subplots(figsize=(17, 5), nrows=3, sharey=True, sharex=True)

ax0.plot(t, obs.anomaly, label="Observations", color="C0")
ax1.plot(t, tide.h, label="Prediction", color="C1")
ax2.plot(t, obs.anomaly - tide.h, label="Residual", color="C2")
fig.legend(ncol=3, loc="upper center");